[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc6_llms/cours/seance1_cours.ipynb)

# Séance 6.1 — Science des données et LLMs

**Cours** · durée : 4h (2h de cours, 2h d'atelier)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- expliquer simplement le fonctionnement et les limites d'un LLM
- distinguer LLM, RAG, tool calling, agent et MCP
- appeler un LLM depuis Python et l'utiliser sur des données textuelles
- vérifier ses prédictions plutôt que les croire sur parole
- donner des fonctions Python au modèle comme outils, et comprendre comment un agent les enchaîne

## Plan du cours

### A) Comprendre les LLMs

- **Qu'est-ce qu'un LLM ?**
  - définition
  - principaux usages
- **Tokens et génération de texte**
  - notion de token
  - prédiction du token suivant
- **Entraînement et inférence**
  - comment le modèle apprend
  - ce qui se passe lorsqu'on l'utilise
- **Prompt et contexte**
  - rôle du prompt
  - fenêtre de contexte
- **Limites des LLMs**
  - informations auxquelles le modèle n'a pas accès
  - hallucinations
  - vérification des réponses
- **LLMs et Data Science**
  - tâches adaptées aux LLMs
  - tâches qu'il vaut mieux laisser à Python
- **Impact environnemental**
  - consommation des data centers
  - impact des usages génératifs et agentiques
  - bonnes pratiques

### B) Utiliser et augmenter un LLM

- **Modèle et interfaces**
  - interface de chat
  - API
- **Pourquoi augmenter un LLM ?**
  - accès limité aux documents, données et systèmes externes
- **RAG**
  - recherche d'informations pertinentes
  - chunks
  - embeddings
- **Tools et function calling**
  - rendre des fonctions disponibles au modèle
  - distinguer le choix de l'outil et son exécution
- **Agents**
  - différence entre tool calling et agent
  - boucle décision → action → observation
  - agents de programmation
- **MCP**
  - difficultés liées à la multiplication des intégrations
  - standardisation de l'accès aux outils et aux données
- **Sécurité et maîtrise des ressources**
  - permissions
  - sandbox
  - tokens et coûts

### C) Intégrer un LLM dans un workflow de Data Science

- **Préparer l'environnement du cours**
  - Google Colab
  - clé API Gemini
  - installation des bibliothèques
  - import du fichier de données
- **Découvrir un modèle NLP spécialisé pré-entraîné**
  - BERT
  - fine-tuning pour l'analyse de sentiment
  - inférence locale dans Colab
- **Classifier des avis avec BERT**
  - prédiction d'une note de 1 à 5 étoiles
  - conversion en sentiment positif, négatif ou neutre
  - évaluation globale
  - évaluation selon le type d'avis
- **Comparer BERT à un LLM génératif**
  - classification avec Gemini
  - appels API
  - comparaison des prédictions
  - analyse particulière des avis sarcastiques
- **Comprendre les désaccords entre modèles**
  - cas correctement et incorrectement classés
  - rôle du contexte
  - limites de la comparaison
- **Donner une fonction Python au LLM**
  - déclaration d'un outil
  - tool calling
  - exécution réelle de la fonction par Python
- **Construire un véritable agent**
  - plusieurs outils
  - boucle multi-étapes
  - choix d'une nouvelle action à partir du résultat précédent
- **Comparer les approches**
  - modèle NLP spécialisé
  - LLM génératif
  - agent
  - Python direct
  - limites et bonnes pratiques

# A) Comprendre les LLMs

## 1. Qu'est-ce qu'un LLM ?

**LLM = Large Language Model**

Un LLM est un modèle capable de traiter et de générer du langage.

Il peut notamment :

- répondre à des questions
- résumer ou reformuler un texte
- traduire
- classifier du texte
- extraire des informations
- produire ou expliquer du code

## 2. Tokens et génération de texte

Un LLM produit une réponse en prédisant progressivement **le prochain token**.

Un **token** est un morceau de texte :

- un mot
- une partie de mot
- un signe de ponctuation
- un symbole

### Exemple simplifié

À partir de :

> « Python est un langage très… »

le modèle peut attribuer différentes probabilités aux tokens suivants :

| Token possible | Probabilité illustrative |
|---|---:|
| `utile` | 30 % |
| `populaire` | 24 % |
| `puissant` | 18 % |
| `simple` | 12 % |

Le modèle choisit un token, l'ajoute au texte, puis recommence.

**Génération :** prompt → token 1 → token 2 → token 3 → … → réponse.

### Conséquence

Le modèle cherche à produire une suite de texte plausible.

**Plausible ne signifie pas forcément vrai.**

## 3. Entraînement et inférence

### Entraînement

Avant que nous utilisions le modèle :

1. de grandes quantités de données sont utilisées
2. le modèle apprend des régularités
3. on obtient un modèle entraîné

Le modèle apprend notamment des relations entre les mots,
des structures de phrases et des formes fréquentes de code.

### Inférence

Lorsque nous utilisons ensuite le modèle :

1. nous envoyons un prompt
2. le modèle déjà entraîné le traite
3. il génère une réponse

Dans ce cours, nous faisons principalement de **l'inférence**.

## 4. Prompt et contexte

Le **prompt** est l'instruction envoyée au modèle.

Prompt vague :

> Analyse ce commentaire.

Prompt plus précis :

> Classe ce commentaire en positif ou négatif.  
> Réponds uniquement par `1` pour positif ou `0` pour négatif.

Un prompt peut préciser :

1. la tâche
2. le contexte
3. les contraintes
4. le format attendu

### La fenêtre de contexte

Le **contexte** correspond à l'ensemble des informations disponibles
pour produire la réponse.

Il peut notamment contenir :

- les instructions données au modèle
- la question de l'utilisateur
- l'historique de la conversation
- des documents
- des données
- les résultats d'outils

La quantité de contexte utilisable est limitée :
on parle de **fenêtre de contexte**.

Cela explique pourquoi, face à un très grand document,
on préfère souvent rechercher les passages pertinents
plutôt que transmettre l'intégralité du document.

## 5. Ce qu'un LLM sait et ne sait pas

Un LLM peut connaître :

- la syntaxe Python
- des méthodes Pandas
- des connaissances générales
- des structures fréquentes de code

Mais il ne connaît pas automatiquement :

- le fichier présent sur votre ordinateur
- le contenu du DataFrame `df`
- vos documents internes
- une base privée
- le résultat d'un calcul qui n'a pas été exécuté

Exemple :

> Quelle est la moyenne de la colonne `age` de mon DataFrame ?

Sans accès à `df`, le modèle ne peut pas connaître le vrai résultat.

## 6. Hallucinations et vérification

Une **hallucination** est une information incorrecte ou inventée
présentée comme si elle était vraie.

Quelques causes :

- information absente
- contexte insuffisant
- question ambiguë
- génération d'une réponse plausible malgré l'incertitude

### Bonnes pratiques

- donner le contexte nécessaire
- utiliser des sources
- utiliser Python pour les calculs exacts
- tester le code généré
- vérifier les résultats importants

> **Le LLM propose ; Python calcule ; l'humain vérifie.**

## 7. Usages en programmation et en Data Science

### Usages adaptés au LLM

- expliquer une erreur
- résumer des commentaires
- classifier du texte
- extraire des informations
- proposer du code
- expliquer un résultat

### Tâches à confier directement au code

- compter des observations
- calculer une moyenne
- faire un `groupby()`
- estimer une régression
- calculer une métrique reproductible

L'intérêt est souvent de **combiner** LLM et outils classiques.

## 8. Impact environnemental de l'IA générative et agentique

L'entraînement et l'utilisation des modèles nécessitent des **data centers**
contenant de nombreux serveurs et processeurs.

Quelques ordres de grandeur publiés par l'International energy agency (IEA) :

- environ **485 TWh** d'électricité consommés par les data centers dans le monde en 2025
- environ **950 TWh** projetés en 2030
- soit environ **3 % de la demande mondiale d'électricité** en 2030
- la consommation des data centers principalement dédiés à l'IA
  a progressé d'environ **50 % en 2025**

> Ces chiffres concernent les data centers et non uniquement les LLMs.
> L'IA représente une part croissante de cette consommation.

### Pourquoi les usages agentiques peuvent-ils consommer plus ?

Une génération simple peut se limiter à :

**question → LLM → réponse**

Une tâche agentique peut au contraire nécessiter plusieurs appels successifs :

**LLM → outil → LLM → autre outil → LLM → réponse**

L'IEA indique que certains usages complexes
tels que le raisonnement avancé, la génération vidéo ou les tâches agentiques
peuvent consommer **des centaines à des milliers de fois plus d'énergie par requête**
qu'une génération de texte simple.

### Bonnes pratiques

- utiliser un modèle adapté à la tâche
- éviter les requêtes inutiles
- limiter la taille du contexte
- demander une réponse courte lorsque cela suffit
- ne transmettre que les passages pertinents
- limiter le nombre d'étapes d'un agent
- utiliser directement Python lorsqu'un simple calcul suffit

> **Utiliser l'outil le plus simple adapté à la tâche réduit
> à la fois les coûts et l'impact environnemental.**

# B) Utiliser et augmenter un LLM

Dans cette partie, **augmenter un LLM** ne signifie pas le réentraîner.

Il s'agit de lui donner des capacités supplémentaires :

| Besoin | Solution |
|---|---|
| Lui fournir une information qu'il ne connaît pas | **RAG** |
| Lui permettre d'utiliser une fonction ou un service | **Tools / function calling** |
| Lui permettre de choisir plusieurs actions successives | **Agent** |

Nous allons introduire ces notions progressivement.

## 1. Modèle et interfaces d'utilisation

Le **LLM** est le modèle.

L'**interface** est la manière dont nous communiquons avec lui.

### Interface de chat

L'utilisateur écrit directement dans une application comme ChatGPT,
Claude ou Gemini, qui transmet sa demande au modèle.

**Utilisateur → interface de chat → LLM → réponse**

### API

Un programme peut aussi envoyer directement une demande au modèle :

**Programme Python → API → LLM → réponse renvoyée au programme**

L'API permet donc d'intégrer le modèle directement dans un programme.

### À quoi sert l'API ?

Avec une interface de chat, l'utilisateur copie et lit les réponses manuellement.

Avec une API, un programme peut :

- envoyer automatiquement plusieurs textes
- récupérer les réponses dans des variables
- les ajouter à un DataFrame
- automatiser un traitement

> **Chat et API sont donc deux façons d'accéder au modèle.**

## 2. Pourquoi un LLM seul ne suffit-il pas toujours ?

Le LLM ne possède pas automatiquement :

- nos documents
- nos fichiers
- notre DataFrame
- nos bases privées
- le résultat d'un calcul
- la possibilité d'agir sur un service externe

Deux problèmes différents apparaissent :

### Information manquante

> Quelle est la politique de télétravail de mon entreprise ?

### Action ou calcul nécessaire

> Quelle est la moyenne de `sales` dans mon DataFrame ?

Ces deux problèmes appellent des solutions différentes.

## 3. RAG : apporter le bon contexte

**RAG = Retrieval-Augmented Generation**

Le principe est simple :

1. l'utilisateur pose une question
2. le système recherche les passages pertinents dans des documents
3. ces passages sont ajoutés au contexte
4. le LLM produit sa réponse à partir de ce contexte

Le modèle ne doit donc plus « deviner » une information :
on lui fournit le passage utile.

> **RAG = rechercher le bon contexte avant de demander au LLM de répondre.**

### Comment fonctionne un RAG ?

Un système RAG réel utilise souvent trois notions.

### Chunks

Un grand document est découpé en petits passages appelés **chunks**.

Par exemple, un document peut être séparé en :

- chunk 1 : introduction
- chunk 2 : politique de télétravail
- chunk 3 : congés
- chunk 4 : remboursement des transports

### Embeddings

Chaque passage peut être représenté numériquement
afin de comparer son sens à celui de la question.

### Retrieval

Le système sélectionne ensuite les passages les plus pertinents
et les ajoute au contexte du LLM.

> **À retenir : on recherche d'abord l'information utile, puis on appelle le LLM.**

## 4. Tools / function calling : donner des capacités au LLM

Prenons :

> Quelle est la moyenne de `sales` ?

Python peut faire le calcul :

```python
def mean_sales():
    return df["sales"].mean()
```

On peut indiquer au modèle que cette fonction est disponible.

Le **tool calling** permet alors au modèle de demander son utilisation.

### Comment un LLM utilise-t-il un outil ?

Le LLM n'a pas accès automatiquement à toutes nos fonctions.

L'application lui indique les fonctions qu'il est autorisé à utiliser.

Le processus est alors :

1. l'utilisateur pose une question
2. le LLM détermine si une fonction est nécessaire
3. il demande l'utilisation de cette fonction
4. le programme Python exécute la fonction
5. le résultat est renvoyé au LLM
6. le LLM formule la réponse finale

> **Le LLM choisit l'outil, mais c'est le programme qui l'exécute.**

### Comment s'articulent chat, API, tools et agents ?

- **LLM** = le modèle
- **Chat / API** = des façons d'utiliser le modèle
- **Tool calling** = permettre au modèle de demander l'utilisation d'une fonction
- **Agent** = système utilisant un LLM et des outils pour accomplir une tâche

Un chatbot moderne peut lui-même utiliser des outils en arrière-plan :
recherche web, fichiers, code, API, etc.

## 5. Du tool calling à l'agent

Pour une question simple, un seul outil peut suffire.

Mais une tâche plus complexe peut nécessiter plusieurs étapes.

Exemple :

> Analyse la base et identifie les principales différences entre les sources.

Il peut être nécessaire de :

1. calculer une première statistique
2. observer le résultat
3. choisir une nouvelle analyse
4. observer le nouveau résultat
5. décider de s'arrêter

La question devient :

> **Que faut-il faire ensuite ?**

## 6. Qu'est-ce qu'un agent ?

Un agent simple combine généralement :

- un **LLM**
- des **instructions**
- des **outils**
- une **boucle de décision**

### Fonctionnement général

1. le modèle comprend la tâche
2. il choisit une action
3. un outil est exécuté
4. le modèle observe le résultat
5. il décide s'il possède assez d'informations
6. sinon, il choisit une nouvelle action
7. lorsqu'il a terminé, il produit la réponse finale

### Différence essentielle

| Tool calling | Agent |
|---|---|
| Le modèle demande l'utilisation d'un outil | Le modèle peut utiliser plusieurs outils successivement |
| Un appel peut suffire | Le résultat d'une action peut influencer l'action suivante |
| Pas nécessairement de boucle | Boucle **décision → action → observation** |

> **Tool calling = utiliser un outil.  
> Agent = utiliser des outils dans une boucle de décision.**

## 7. Agents de programmation

Des agents comme **Codex**, **Claude Code** ou **Gemini CLI**
appliquent cette logique au développement logiciel.

Selon les permissions accordées, ils peuvent :

- lire plusieurs fichiers
- rechercher du code
- proposer des modifications
- lancer des commandes
- lancer des tests
- observer les résultats puis poursuivre la tâche

Ils peuvent être utilisés directement depuis un terminal ou un environnement de développement.

### Exemple dans un terminal

Imaginons un dossier contenant :

- `analyse.py`
- `data.csv`
- `README.md`

Un agent de programmation peut recevoir :

> Explique pourquoi `analyse.py` produit une erreur et propose une correction.

Il peut alors consulter plusieurs fichiers et utiliser ses outils
pour avancer dans la tâche.

Pour le cours, nous n'avons pas besoin d'installer ces agents :
nous construisons notre propre exemple simple dans Colab.

## 8. Connecter un agent à plusieurs outils : difficultés d'intégration

Imaginons un agent qui doit :

- récupérer les ventes
- vérifier les stocks
- envoyer un e-mail

Il doit communiquer avec plusieurs services.

| Besoin | Service possible |
|---|---|
| Récupérer les ventes | API de la base de ventes |
| Vérifier les stocks | API du logiciel de stocks |
| Envoyer le résumé | API du service d'e-mail |

Chaque API peut avoir :

- une authentification différente
- des fonctions différentes
- des paramètres différents
- un format de réponse différent

Plus on ajoute d'outils,
plus il faut développer et maintenir d'intégrations spécifiques.

## 9. MCP : standardiser l'accès à plusieurs outils et services

**MCP = Model Context Protocol**

MCP est un standard ouvert introduit par **Anthropic en novembre 2024**.

Son objectif est de proposer une manière commune
de présenter des outils et des données aux applications utilisant des LLMs.

Au lieu que l'application gère directement chaque intégration de manière différente,
elle peut communiquer avec des **serveurs MCP** selon un protocole commun.

Exemple :

- l'agent communique avec un serveur MCP « ventes »
- un autre serveur MCP expose les stocks
- un autre expose l'envoi d'e-mails

MCP a ensuite été adopté par de nombreux outils et écosystèmes
au-delà d'Anthropic.

### MCP ne remplace pas les API

Les services externes peuvent toujours utiliser leurs propres API.

Par exemple :

1. l'agent demande l'outil `get_stock()` via MCP
2. le serveur MCP reçoit la demande
3. le serveur peut appeler l'API du logiciel de stocks
4. le résultat est renvoyé à l'agent

| Notion | Rôle |
|---|---|
| **API** | Permet de communiquer avec un service particulier |
| **MCP** | Standardise la manière dont des outils et ressources sont présentés aux applications utilisant des LLMs |

MCP ajoute donc une **couche standardisée** :
il ne supprime pas nécessairement les API utilisées en arrière-plan.

### Que peut exposer un serveur MCP ?

Un serveur MCP peut notamment exposer :

| Élément | Rôle | Exemple |
|---|---|---|
| **Tools** | Actions que l'application peut demander | `get_sales()`, `send_email()` |
| **Resources** | Informations consultables | fichier, document, données |
| **Prompts** | Instructions réutilisables | modèle de résumé |

> MCP facilite la connexion entre agents, outils et données,
> mais ne remplace pas les règles de sécurité ou d'authentification.

## 10. Sécurité, permissions et maîtrise des ressources

Plus un système peut agir, plus il faut le contrôler.

Risques possibles :

- lecture de données sensibles
- modification ou suppression de fichiers
- commande dangereuse
- fuite d'une clé API
- appels répétés au modèle
- consommation excessive de tokens

### Limiter les permissions

Un agent doit recevoir uniquement les capacités nécessaires.

Par exemple, un agent chargé d'analyser un fichier peut avoir besoin de :

- lire le fichier
- exécuter du Python

Il n'a pas forcément besoin de :

- supprimer des fichiers
- envoyer des e-mails
- accéder librement à Internet

> **Principe : Donner le minimum de permissions nécessaires.**

### Sandbox

Une **sandbox** est un environnement isolé
dans lequel le code peut être exécuté avec des accès limités.

Une sandbox peut par exemple limiter :

- les fichiers accessibles
- les fichiers modifiables
- les commandes autorisées
- l'accès au réseau

Elle permet de réduire les conséquences d'une mauvaise action.

| Notion | Question |
|---|---|
| **Permissions** | Qu'est-ce que l'agent est autorisé à demander ? |
| **Sandbox** | Jusqu'où l'action exécutée peut-elle réellement agir ? |

Les deux mécanismes sont complémentaires.

### Tokens et coûts

Les appels à un LLM consomment des **tokens**.

### Tokens d'entrée

Ils peuvent inclure :

- le prompt
- l'historique
- les documents
- les résultats des outils

### Tokens de sortie

Ils correspondent au contenu généré par le modèle.

Un agent peut multiplier les appels au modèle :

**LLM → outil → LLM → autre outil → LLM**

Cela peut augmenter :

- le nombre de tokens
- le coût financier
- le temps d'exécution
- l'impact environnemental

### Comment limiter les coûts ?

- fournir uniquement le contexte nécessaire
- utiliser le RAG plutôt que transmettre un document entier
- demander des réponses courtes
- choisir un modèle adapté à la tâche
- limiter le nombre maximal d'étapes
- éviter les appels répétés au même outil
- utiliser Python directement pour les calculs simples

Une limite d'étapes permet aussi d'éviter qu'un agent répète
inutilement les mêmes actions, par exemple :

**outil A → outil B → outil A → outil B → …**

### Synthèse de la partie B

| Notion | À quoi sert-elle ? |
|---|---|
| **LLM** | Comprendre et générer du langage |
| **Chat / API** | Accéder au modèle |
| **RAG** | Lui apporter un contexte externe pertinent |
| **Tool calling** | Lui permettre de demander l'utilisation d'une fonction |
| **Agent** | Choisir et enchaîner plusieurs actions |
| **MCP** | Standardiser l'accès aux outils et aux ressources |

# C) Intégrer un LLM dans un workflow de Data Science

Nous allons travailler sur un même problème pendant toute cette partie du cours

> Comment différentes approches comprennent-elles des avis clients qui contiennent parfois du sarcasme ?

Nous allons comparer trois niveaux

1. un modèle NLP spécialisé pré-entraîné
2. un LLM génératif utilisé via une API
3. un agent capable d'utiliser plusieurs outils d'analyse

L'objectif n'est pas de montrer qu'une méthode est toujours meilleure qu'une autre

L'objectif est de comprendre leurs différences et d'identifier les situations dans lesquelles la compréhension du contexte apporte une valeur supplémentaire

## 1. Préparer l'environnement du cours

Nous utiliserons deux modèles différents

| Modèle | Utilisation dans le cours | Où est-il exécuté ? |
|---|---|---|
| **BERT spécialisé en sentiment** | Classification d'avis clients | Dans l'environnement Colab |
| **Gemini 3.5 Flash-Lite** | Classification générative, tool calling et agent | Sur les serveurs de Google via une API |

Il faut donc préparer à la fois l'environnement Python de Colab et l'accès à l'API Gemini

### 1.1. Ouvrir le notebook dans Google Colab

1. ouvrir https://colab.research.google.com/
2. choisir **Importer**
3. sélectionner le notebook fourni pour le cours
4. attendre l'ouverture du notebook
5. cliquer sur **Connecter** en haut à droite

Colab crée alors un environnement distant dans lequel Python exécutera les cellules du notebook

### 1.2. Créer une clé API Gemini

1. ouvrir https://aistudio.google.com/
2. se connecter avec son compte Google
3. ouvrir la section de création des clés API
4. créer une nouvelle clé
5. copier la clé

Une clé API est personnelle

Elle ne doit pas être écrite directement dans le notebook, partagée avec un autre étudiant ou publiée sur GitHub

### 1.3. Enregistrer la clé dans les Secrets de Colab

1. ouvrir le panneau **Secrets** dans la barre de gauche
2. ajouter un nouveau secret
3. utiliser exactement le nom `GEMINI_API_KEY`
4. coller la clé dans la valeur du secret
5. autoriser le notebook à accéder à ce secret

Le notebook pourra ensuite utiliser la clé sans l'afficher

### 1.4. Installer les bibliothèques

Nous utilisons deux bibliothèques principales

- `transformers` pour charger le modèle NLP pré-entraîné
- `google-genai` pour communiquer avec l'API Gemini

Nous fixons la version de `google-genai` utilisée dans le cours afin de conserver un environnement commun

In [ ]:
%pip install -q "transformers>=4.45,<5" "google-genai==2.9.0"

Si Colab demande de redémarrer la session après l'installation, redémarrez-la puis reprenez à partir de la cellule suivante

### 1.5. Importer le fichier de données

Le matériel du cours contient le fichier

`avis.csv`

Le fichier contient environ 50 avis clients fictifs avec un équilibre entre

- avis positifs
- avis négatifs non sarcastiques
- avis négatifs sarcastiques

Pour chaque avis, la base contient notamment le texte, la note attribuée par le client et le sentiment réel

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc6_llms/data/"

In [ ]:
df = pd.read_csv(BASE + "avis.csv")

df.head()

In [ ]:
print("Nombre d'avis :", len(df))
print()
print(df["type"].value_counts())

Les principales colonnes sont

| Colonne | Signification |
|---|---|
| `text` | Texte de l'avis |
| `rating` | Note attribuée par le client entre 1 et 5 |
| `type` | Avis positif, négatif ou sarcastique |
| `sentiment` | Sentiment réel positif ou négatif |
| `sarcasm` | Indicateur de présence de sarcasme |